In [2]:
import pandas as pd
import numpy as np
import joblib

print("Libraries loaded.")

Libraries loaded.


In [3]:
# Load structural models, Load All Trained Models

complexity_model = joblib.load("../models/model_complexity.pkl")
priority_model = joblib.load("../models/model_client_priority.pkl")
team_model = joblib.load("../models/model_team_experience.pkl")

# Load regression models
delay_model = joblib.load("../models/model_delay_rate.pkl")
delay_encoder = joblib.load("../models/delay_encoder.pkl")

completion_model = joblib.load("../models/model_completion_days.pkl")

# Load label decoders
target_encoders = joblib.load("../models/target_label_encoders.pkl")

print("All models loaded successfully.")

All models loaded successfully.


In [4]:
#Enrichment Function

def enrich_project(project_title, project_type, deadline_days, base_revenue):
    
    # -----------------------------
    # STEP 1: Prepare base input
    # -----------------------------
    base_input = pd.DataFrame([{
        "project_title": project_title,
        "project_type": project_type,
        "deadline_days": deadline_days,
        "base_revenue": base_revenue
    }])
    
    # -----------------------------
    # STEP 2: Predict Structural Features
    # -----------------------------
    pred_complexity = complexity_model.predict(base_input)
    pred_priority = priority_model.predict(base_input)
    pred_team = team_model.predict(base_input)
    
    # Decode back to original labels
    complexity = target_encoders["complexity_level"].inverse_transform(pred_complexity)[0]
    client_priority = target_encoders["client_priority"].inverse_transform(pred_priority)[0]
    team_experience = target_encoders["team_experience_level"].inverse_transform(pred_team)[0]
    
    # -----------------------------
    # STEP 3: Predict Delay Rate
    # -----------------------------
    delay_input = pd.DataFrame([{
        "project_type": project_type,
        "complexity_level": pred_complexity[0],
        "team_experience_level": pred_team[0],
        "client_priority": pred_priority[0]
    }])
    
    delay_input_encoded = delay_encoder.transform(delay_input)
    predicted_delay_rate = delay_model.predict(delay_input_encoded)[0]
    
    # -----------------------------
    # STEP 4: Predict Completion Days
    # -----------------------------
    completion_input = pd.DataFrame([{
        "deadline_days": deadline_days,
        "complexity_level": pred_complexity[0],
        "team_experience_level": pred_team[0],
        "historical_delay_rate": predicted_delay_rate
    }])
    
    predicted_completion_days = completion_model.predict(completion_input)[0]
    
    # -----------------------------
    # STEP 5: Derive Final Metrics
    # -----------------------------
    delay_days = max(0, round(predicted_completion_days - deadline_days))
    
    completed_on_time = 1 if delay_days == 0 else 0
    
    # Revenue deduction logic
    penalty_per_day = 0.05  # 5% penalty per delay day
    penalty = base_revenue * penalty_per_day * delay_days
    final_revenue = max(0, base_revenue - penalty)
    
    # -----------------------------
    # Final Output
    # -----------------------------
    enriched_record = {
        "project_title": project_title,
        "project_type": project_type,
        "deadline_days": deadline_days,
        "base_revenue": base_revenue,
        
        "complexity_level": complexity,
        "client_priority": client_priority,
        "team_experience_level": team_experience,
        
        "predicted_delay_rate": round(predicted_delay_rate, 2),
        "predicted_completion_days": round(predicted_completion_days, 2),
        "delay_days": delay_days,
        "completed_on_time": completed_on_time,
        "final_revenue_realized": round(final_revenue, 2)
    }
    
    return enriched_record

In [5]:
result = enrich_project(
    project_title="AI Fraud Detection Engine",
    project_type="AI Development",
    deadline_days=4,
    base_revenue=150000
)

result

{'project_title': 'AI Fraud Detection Engine',
 'project_type': 'AI Development',
 'deadline_days': 4,
 'base_revenue': 150000,
 'complexity_level': 'Very High',
 'client_priority': 'Very High',
 'team_experience_level': 'Senior Team',
 'predicted_delay_rate': np.float64(0.43),
 'predicted_completion_days': np.float64(4.98),
 'delay_days': 1,
 'completed_on_time': 0,
 'final_revenue_realized': 142500.0}

In [7]:
pip install fastapi uvicorn scikit-learn pandas joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\dipes\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip
